# 后效大盘分布分析 —— 消耗类

**数据来源**：`ad_callback_log_from_ad_log_full`，通过 Spark 直连 Hive 取数  
**分析维度**：`visitor_id`  
**分箱方式**：等频分箱（Quantile），10个分箱  
**覆盖口径**：总消耗 / 正常流量消耗 / 作弊流量消耗 / 内循环消耗 / 外循环CID消耗 / 线索消耗 / 下载消耗

---
**表结构说明**：
- 同一个 `llsid`（一次广告曝光/点击）会产生多行记录
  - **计费行**：`action_type = charge_action_type`，有 `cost_total`，用于消耗类分析
  - **事件行**：`action_type` 为各种 `EVENT_*`，`cost_total = 0`，用于后效事件类分析
- 本 notebook 只处理消耗类，Hive 层直接过滤 `action_type = charge_action_type` 取计费行

## Step 1：配置

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession

P_DATE = '20250101'
N_BINS = 10

INCYCLE_OCPC_TYPES     = "'AD_MERCHANT_ROAS','EVENT_ORDER_PAIED','AD_STOREWIDE_ROAS','AD_MERCHANT_T7_ROI','AD_FANS_TOP_ROI'"
EXCYCLE_CID_OCPC_TYPES = "'EVENT_ORDER_SUBMIT','AD_CID_ROAS','CID_ROAS','CID_EVENT_ORDER_PAID'"
CLUE_OCPC_TYPES        = "'AD_LANDING_PAGE_FORM_SUBMITTED','EVENT_VALID_CLUES','AD_EFFECTIVE_CUSTOMER_ACQUISITION','EVENT_INTENTION_CONFIRMED','LEADS_SUBMIT','EVENT_PHONE_GET_THROUGH'"
DOWNLOAD_OCPC_TYPES    = "'AD_CONVERSION','AD_PURCHASE','AD_PURCHASE_CONVERSION','AD_ROAS','AD_SEVEN_DAY_ROAS','AD_ITEM_DOWNLOAD_COMPLETED','AD_ITEM_CLICK_DOWNLOAD'"

## Step 2：初始化 Spark

In [ ]:
spark = SparkSession.builder.appName("cost_distribution_analysis").enableHiveSupport().getOrCreate()
print("Spark 初始化完成")

## Step 3：从 Hive 取计费行并聚合到 visitor_id 维度

在 Hive 层直接做 `GROUP BY visitor_id`，只拉聚合结果到 pandas，避免全量数据落本地。

In [ ]:
sql_cost = f"""
SELECT
    visitor_id,
    SUM(cost_total) / 1000.0                                                                          AS total_cost_yuan,
    SUM(IF(is_for_report_engine = true,  cost_total, 0)) / 1000.0                                     AS normal_cost_yuan,
    SUM(IF(is_for_report_engine = false, cost_total, 0)) / 1000.0                                     AS spam_cost_yuan,
    SUM(IF(ocpc_action_type IN ({INCYCLE_OCPC_TYPES}),     cost_total, 0)) / 1000.0                   AS incycle_cost_yuan,
    SUM(IF(ocpc_action_type IN ({EXCYCLE_CID_OCPC_TYPES}), cost_total, 0)) / 1000.0                   AS excycle_cid_cost_yuan,
    SUM(IF(ocpc_action_type IN ({CLUE_OCPC_TYPES}),        cost_total, 0)) / 1000.0                   AS clue_cost_yuan,
    SUM(IF(ocpc_action_type IN ({DOWNLOAD_OCPC_TYPES}),    cost_total, 0)) / 1000.0                   AS download_cost_yuan
FROM ks_origin_ad_log.ad_callback_log_from_ad_log_full
WHERE p_date = '{P_DATE}'
  AND is_duplicate = false
  AND is_retry = false
  AND action_type = charge_action_type
GROUP BY visitor_id
"""

print("正在从 Hive 取数...")
df = spark.sql(sql_cost).toPandas()
print(f"取数完成，visitor_id 数: {len(df):,}")
df.head()

## Step 4：工具函数

In [ ]:
def qcut_distribution(df, value_col, n_bins=N_BINS):
    s = df[value_col].dropna()
    s = s[s > 0]
    if s.empty:
        print(f"  [WARN] {value_col}: 无有效数据")
        return pd.DataFrame()
    bins = pd.qcut(s, q=n_bins, duplicates='drop')
    result = (
        s.groupby(bins)
        .agg(
            cnt='count',
            min_val='min',
            max_val='max',
            avg_val='mean',
            total_val='sum',
        )
        .reset_index()
    )
    result[value_col] = result[value_col].astype(str)
    result['pct%'] = (result['cnt'] / result['cnt'].sum() * 100).round(2)
    result.columns = ['分箱区间', '用户数cnt', '最小值(元)', '最大值(元)', '均值(元)', '总消耗(元)', '占比%']
    return result


def show_distribution(result, metric_name, save_csv=True):
    print(f"\n{'='*65}")
    print(f"  {metric_name}  （visitor_id维度，等频{N_BINS}箱）")
    print(f"{'='*65}")
    if result.empty:
        return
    print(result.to_string(index=False))
    if save_csv:
        fname = f"dist_{metric_name}.csv"
        result.to_csv(fname, index=False)
        print(f"  => 已保存: {fname}")

## Step 5：各消耗口径分布分析

### 5.1 总消耗（total_cost_yuan）

In [ ]:
show_distribution(qcut_distribution(df, 'total_cost_yuan'), 'total_cost_yuan')

### 5.2 正常流量消耗（normal_cost_yuan）
> `is_for_report_engine = true`

In [ ]:
show_distribution(qcut_distribution(df, 'normal_cost_yuan'), 'normal_cost_yuan')

### 5.3 作弊流量消耗（spam_cost_yuan）
> `is_for_report_engine = false`

In [ ]:
show_distribution(qcut_distribution(df, 'spam_cost_yuan'), 'spam_cost_yuan')

### 5.4 内循环消耗（incycle_cost_yuan）
> `ocpc_action_type IN ('AD_MERCHANT_ROAS','EVENT_ORDER_PAIED','AD_STOREWIDE_ROAS','AD_MERCHANT_T7_ROI','AD_FANS_TOP_ROI')`

In [ ]:
show_distribution(qcut_distribution(df, 'incycle_cost_yuan'), 'incycle_cost_yuan')

### 5.5 外循环CID消耗（excycle_cid_cost_yuan）
> `ocpc_action_type IN ('EVENT_ORDER_SUBMIT','AD_CID_ROAS','CID_ROAS','CID_EVENT_ORDER_PAID')`

In [ ]:
show_distribution(qcut_distribution(df, 'excycle_cid_cost_yuan'), 'excycle_cid_cost_yuan')

### 5.6 线索消耗（clue_cost_yuan）
> `ocpc_action_type IN ('AD_LANDING_PAGE_FORM_SUBMITTED','EVENT_VALID_CLUES','AD_EFFECTIVE_CUSTOMER_ACQUISITION','EVENT_INTENTION_CONFIRMED','LEADS_SUBMIT','EVENT_PHONE_GET_THROUGH')`

In [ ]:
show_distribution(qcut_distribution(df, 'clue_cost_yuan'), 'clue_cost_yuan')

### 5.7 下载消耗（download_cost_yuan）
> `ocpc_action_type IN ('AD_CONVERSION','AD_PURCHASE','AD_PURCHASE_CONVERSION','AD_ROAS','AD_SEVEN_DAY_ROAS','AD_ITEM_DOWNLOAD_COMPLETED','AD_ITEM_CLICK_DOWNLOAD')`

In [ ]:
show_distribution(qcut_distribution(df, 'download_cost_yuan'), 'download_cost_yuan')

## Step 6：汇总总览（p50 / p90 / p99）

In [ ]:
cost_cols = [
    'total_cost_yuan', 'normal_cost_yuan', 'spam_cost_yuan',
    'incycle_cost_yuan', 'excycle_cid_cost_yuan', 'clue_cost_yuan', 'download_cost_yuan'
]

summary = []
for col in cost_cols:
    s = df[col].dropna()
    s = s[s > 0]
    if s.empty:
        summary.append({'口径': col, '有效UV数': 0})
        continue
    summary.append({
        '口径': col,
        '有效UV数': len(s),
        '总消耗(元)': round(s.sum(), 2),
        '人均消耗(元)': round(s.mean(), 4),
        'p50': round(s.quantile(0.5), 4),
        'p90': round(s.quantile(0.9), 4),
        'p99': round(s.quantile(0.99), 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('cost_summary.csv', index=False)
print("\n=> 已保存: cost_summary.csv")